# Notebook 05 - Alineacion canonica NT Inga vs Reina-Valera 1909

**Entrega 2.** Junta los dos jsonl estructurados por la tupla
`(libro, capitulo, versiculo)` y produce `datos/nt_paralelo.jsonl`
con los pares paralelos Inga-espanol listos para entrenamiento.

## Estrategia

INNER JOIN exacto por `(libro, capitulo, versiculo)`. La RV1909 actua como
fuente de verdad sobre la cantidad y orden canonico; los versiculos del NT
Inga que no matcheen contra RV1909 se descartan (probablemente sean falsos
positivos del parser OCR).

Filtros de calidad post-join:
- Eliminar pares con texto_inga o texto_es de menos de 5 palabras
- Eliminar pares donde la relacion de longitudes sea > 5x (probable mismatch)
- Eliminar pares duplicados (mismo verso combinado en NT Inga)


In [1]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INGA = ROOT / "datos" / "nt_inga_estructurado.jsonl"
ES = ROOT / "datos" / "nt_rv1909_estructurado.jsonl"
OUT = ROOT / "datos" / "nt_paralelo.jsonl"

inga = pd.read_json(INGA, lines=True)
es = pd.read_json(ES, lines=True)
print(f"NT Inga: {len(inga):,} registros (unicos: {len(inga.drop_duplicates(subset=['libro','capitulo','versiculo'])):,})")
print(f"NT RV1909: {len(es):,} versiculos")


NT Inga: 6,779 registros (unicos: 6,091)
NT RV1909: 7,955 versiculos


## Join exacto

Se hace INNER JOIN. Los versiculos Inga "verso_combinado" se duplican en el
Notebook 03 con el mismo texto para verso N y N+1, asi que naturalmente
aparecen con dos matches contra RV1909 que tienen versiculos N y N+1
separados. El resultado es que ambos versos quedan con el mismo texto_inga
pero diferente texto_es, lo cual es valido para entrenamiento.


In [2]:
# Deduplicar por seguridad: si el parser Inga produjo el mismo (libro, cap, vers)
# multiples veces (no deberia, pero por si acaso), nos quedamos con el primero.
inga_dedup = inga.drop_duplicates(subset=["libro", "capitulo", "versiculo"], keep="first")
print(f"NT Inga deduplicado: {len(inga_dedup):,}")

paralelo = inga_dedup.merge(es, on=["libro", "capitulo", "versiculo"], how="inner")
print(f"Pares paralelos (join exacto): {len(paralelo):,}")
print(f"Tasa de alineacion vs Inga: {len(paralelo)/len(inga_dedup)*100:.1f}%")
print(f"Tasa de alineacion vs RV1909: {len(paralelo)/len(es)*100:.1f}%")


NT Inga deduplicado: 6,091
Pares paralelos (join exacto): 5,680
Tasa de alineacion vs Inga: 93.3%
Tasa de alineacion vs RV1909: 71.4%


## Filtros de calidad

- **Longitud minima**: ambos textos deben tener >= 5 palabras
- **Relacion de longitudes**: max/min <= 5.0 (descarta mismatches obvios)
- **Texto vacio o ruidoso**: descartar texto que sea principalmente digitos


In [3]:
def es_texto_valido(t: str) -> bool:
    if not isinstance(t, str):
        return False
    palabras = t.split()
    if len(palabras) < 5:
        return False
    # Descartar si > 50% son digitos
    if sum(1 for p in palabras if p.replace(",","").replace(".","").isdigit()) > len(palabras) * 0.5:
        return False
    return True

paralelo["len_inga"] = paralelo.texto_inga.str.split().str.len()
paralelo["len_es"] = paralelo.texto_es.str.split().str.len()
paralelo["ratio"] = paralelo[["len_inga", "len_es"]].max(axis=1) / paralelo[["len_inga", "len_es"]].min(axis=1)

mask = (
    paralelo.texto_inga.apply(es_texto_valido)
    & paralelo.texto_es.apply(es_texto_valido)
    & (paralelo.ratio <= 5.0)
)
filtrado = paralelo[mask].copy()
print(f"Tras filtros de calidad: {len(filtrado):,} pares")
print(f"  Descartados por longitud minima o digitos: {(~paralelo.texto_inga.apply(es_texto_valido) | ~paralelo.texto_es.apply(es_texto_valido)).sum()}")
print(f"  Descartados por ratio > 5x: {(paralelo.ratio > 5.0).sum()}")


Tras filtros de calidad: 5,589 pares
  Descartados por longitud minima o digitos: 37
  Descartados por ratio > 5x: 74


## Estadisticas del corpus paralelo final

In [4]:
print(f"Longitud media Inga: {filtrado.len_inga.mean():.1f} palabras")
print(f"Longitud media RV1909: {filtrado.len_es.mean():.1f} palabras")
print(f"Ratio medio: {filtrado.ratio.mean():.2f}")
print()
print("Distribucion por libro:")
print(filtrado.groupby("libro").size().sort_values(ascending=False).head(10).to_string())


Longitud media Inga: 21.2 palabras
Longitud media RV1909: 20.7 palabras
Ratio medio: 1.44

Distribucion por libro:
libro
Lucas          794
Hechos         704
Mateo          697
Juan           635
Marcos         508
Romanos        346
1 Corintios    304
Apocalipsis    249
Hebreos        226
2 Corintios    180


## Inspeccion: 5 pares aleatorios

In [5]:
muestra = filtrado.sample(5, random_state=42)
for _, row in muestra.iterrows():
    print(f"=== {row.libro} {row.capitulo}:{row.versiculo} ===")
    print(f"  INGA:  {row.texto_inga[:200]}")
    print(f"  ES:    {row.texto_es[:200]}")
    print()


=== Juan 17:15 ===
  INGA:  Mana mañakuikichu: kam, kai alpamanda paikunata llugsichipuai. Tukui mana allilla ruraikunamanda kispichipuangi, paikunamanda mañakuiki.
  ES:    No ruego que los quites del mundo, sino que los guardes del mal.

=== Lucas 6:13 ===
  INGA:  Pakariuraka, tukui paita katiraiagkunata kaiaspa, chunga iskaikunatami agllarka. Chi agllaskakunataka nirka: —Nukami kamkunata kachanakuikichita, nukamanda Alli Willaita willagkuna kangapa.
  ES:    Y como fué de día, llamó á sus discípulos, y escogió doce de ellos, á los cuales también llamó apóstoles:

=== Hechos 17:20 ===
  INGA:  Sutipami nukanchi ñi imaurapas mana uiaskata kam rimakungi. Chimandami munanchi, kam willakuska ima niraiaskata iachangapa—.
  ES:    Porque pones en nuestros oídos unas nuevas cosas: queremos pues saber qué quiere ser esto.

=== Marcos 2:8 ===
  INGA:  Jesuska, chasa iuianakuskata iachaspa, nirkakunata: — Imapatak chasa sungullapi iuianakungichi?
  ES:    Y conociendo luego Jesús en su espíri

## Persistir a JSONL

In [6]:
# Campos finales: idx, libro, capitulo, versiculo, texto_inga, texto_es,
# dialecto (AP - el NT Wycliffe esta principalmente en Alto Putumayo), fuente
salida = filtrado[["libro", "capitulo", "versiculo", "texto_inga", "texto_es"]].copy()
salida["dialecto"] = "AP"
salida["fuente"] = "NT-Wycliffe-RV1909"
salida = salida.reset_index(drop=True)
salida.insert(0, "idx", salida.index)

OUT.parent.mkdir(parents=True, exist_ok=True)
with OUT.open("w", encoding="utf-8") as f:
    for _, row in salida.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")
print(f"Escrito: {OUT.relative_to(ROOT)}  ({len(salida):,} pares)")


Escrito: datos/nt_paralelo.jsonl  (5,589 pares)
